In [51]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import json
import time
import torch
import pandas as pd

PROJECT_ROOT  = Path('/content/drive/MyDrive/E-waste Battery Extraction CV')
DATASET_ROOT  = Path('/content/drive/MyDrive/E-waste Battery Extraction CV/final_dataset')
YOLO_OUT      = PROJECT_ROOT / '01_yolo_models'
METRICS_OUT   = PROJECT_ROOT / '04_metrics_and_visualisations'
METRICS_OUT.mkdir(parents=True, exist_ok=True)

IMG_SIZE  = 640
SEED      = 42
IMG_EXTS  = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

YOLOV8_WEIGHTS  = YOLO_OUT / 'model_03_yolov8n_seg'  / 'weights' / 'best.pt'
YOLO11_WEIGHTS  = YOLO_OUT / 'model_04_yolo11n_seg'  / 'weights' / 'best.pt'

print('YOLOv8n weights exist:', YOLOV8_WEIGHTS.exists())
print('YOLO11n weights exist:', YOLO11_WEIGHTS.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
YOLOv8n weights exist: True
YOLO11n weights exist: True


In [52]:
!pip install -q ultralytics opencv-python pycocotools

In [53]:
import cv2
import math
import numpy as np
from pathlib import Path

def list_images(image_dir):
    image_dir = Path(image_dir)
    return sorted([p for p in image_dir.glob('*') if p.suffix.lower() in IMG_EXTS])

def yolo_seg_label_to_mask(label_path, image_shape):
    h, w = image_shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    label_path = Path(label_path)
    if not label_path.exists():
        return mask
    with open(label_path, 'r') as f:
        lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        coords = np.array(parts[1:], dtype=np.float32)
        if len(coords) % 2 != 0:
            coords = coords[:-1]
        pts = coords.reshape(-1, 2)
        pts[:, 0] *= w
        pts[:, 1] *= h
        pts = np.round(pts).astype(np.int32)
        if len(pts) >= 3:
            cv2.fillPoly(mask, [pts], 1)
    return mask

def mask_iou(pred, gt, eps=1e-7):
    pred, gt = pred.astype(bool), gt.astype(bool)
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    if union == 0:
        return 1.0 if inter == 0 else 0.0
    return float(inter / (union + eps))

def mask_precision_recall(pred, gt, eps=1e-7):
    pred, gt = pred.astype(bool), gt.astype(bool)
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()
    return float(tp / (tp + fp + eps)), float(tp / (tp + fn + eps))

def centroid_error(pred, gt):
    def centroid(m):
        ys, xs = np.where(m > 0)
        if len(xs) == 0:
            return None
        return float(xs.mean()), float(ys.mean())
    pc, gc = centroid(pred), centroid(gt)
    if pc is None or gc is None:
        return np.nan
    return float(math.sqrt((pc[0]-gc[0])**2 + (pc[1]-gc[1])**2))

In [54]:
def evaluate_ultralytics_model(model_label, weights_path, split='test', conf=0.1):
    weights_path = Path(weights_path)
    assert weights_path.exists(), f'Missing weights: {weights_path}'

    model = YOLO(str(weights_path))
    image_dir = DATASET_ROOT / 'images' / split
    label_dir = DATASET_ROOT / 'labels' / split
    image_paths = list_images(image_dir)
    print(f'\n{model_label}: evaluating {len(image_paths)} images from {split} at conf={conf}')

    ious, precisions, recalls, centroid_errs = [], [], [], []
    total_time = 0.0
    n_detected = 0

    for img_path in image_paths:
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            continue
        h, w = img_bgr.shape[:2]

        gt_mask = yolo_seg_label_to_mask(
            label_dir / f'{img_path.stem}.txt', img_bgr.shape
        )

        t0 = time.perf_counter()
        result = model.predict(
            source=str(img_path),
            conf=conf,
            imgsz=IMG_SIZE,
            verbose=False
        )[0]
        total_time += time.perf_counter() - t0

        pred_mask = np.zeros((h, w), dtype=np.uint8)

        if result.masks is not None and result.masks.xy is not None and len(result.masks.xy) > 0:
            n_detected += 1

            # If box confidence scores are available, take only the best one.
            # Otherwise merge all masks (handles cases where scores are all low).
            if (result.boxes is not None
                    and len(result.boxes) > 0
                    and result.boxes.conf is not None):
                best_idx = int(result.boxes.conf.argmax())
                polys = [result.masks.xy[best_idx]]
            else:
                polys = result.masks.xy

            for poly in polys:
                if len(poly) >= 3:
                    pts = np.round(poly).astype(np.int32)
                    cv2.fillPoly(pred_mask, [pts], 1)

        iou = mask_iou(pred_mask, gt_mask)
        p, r = mask_precision_recall(pred_mask, gt_mask)
        ce = centroid_error(pred_mask, gt_mask)

        ious.append(iou)
        precisions.append(p)
        recalls.append(r)
        centroid_errs.append(ce)

    latency_ms = (total_time / max(1, len(image_paths))) * 1000.0

    # Read mAP from training results.csv using best mask mAP50 epoch
    run_dir = weights_path.parent.parent
    csv_candidates = list(run_dir.rglob('results.csv'))
    map50 = map5095 = box_map50 = box_map5095 = np.nan

    if csv_candidates:
        df_res = pd.read_csv(
            sorted(csv_candidates, key=lambda p: p.stat().st_mtime)[-1]
        )
        col_m50 = 'metrics/mAP50(M)'
        if col_m50 in df_res.columns and df_res[col_m50].notna().any():
            best_row = df_res.loc[df_res[col_m50].idxmax()]
        else:
            best_row = df_res.iloc[-1]

        def get_any(names):
            for n in names:
                if n.strip() in best_row.index:
                    val = float(best_row[n.strip()])
                    return val if not np.isnan(val) else np.nan
            return np.nan

        map50       = get_any(['metrics/mAP50(M)'])
        map5095     = get_any(['metrics/mAP50-95(M)'])
        box_map50   = get_any(['metrics/mAP50(B)', 'metrics/mAP_0.5'])
        box_map5095 = get_any(['metrics/mAP50-95(B)', 'metrics/mAP_0.5:0.95'])
    else:
        print(f'  No results.csv found in {run_dir}')

    mean_iou        = float(np.nanmean(ious))        if ious        else np.nan
    mean_precision  = float(np.nanmean(precisions))  if precisions  else np.nan
    mean_recall     = float(np.nanmean(recalls))     if recalls     else np.nan
    mean_ce         = float(np.nanmean(centroid_errs)) if centroid_errs else np.nan

    summary = {
        'Model':                        model_label,
        'Split':                        split,
        'Conf threshold':               conf,
        'Num Images':                   len(image_paths),
        'Num Detected':                 n_detected,
        'Mean Mask IoU':                round(mean_iou, 4),
        'Mask Precision':               round(mean_precision, 4),
        'Mask Recall':                  round(mean_recall, 4),
        'Centroid Error (px)':          round(mean_ce, 2) if not np.isnan(mean_ce) else np.nan,
        'Latency ms/img':               round(latency_ms, 2),
        'FPS':                          round(1000.0 / latency_ms, 1) if latency_ms > 0 else np.nan,
        'Mask mAP50 (training csv)':    round(map50, 5)       if not np.isnan(map50)       else np.nan,
        'Mask mAP50-95 (training csv)': round(map5095, 5)     if not np.isnan(map5095)     else np.nan,
        'Box mAP50 (training csv)':     round(box_map50, 5)   if not np.isnan(box_map50)   else np.nan,
        'Box mAP50-95 (training csv)':  round(box_map5095, 5) if not np.isnan(box_map5095) else np.nan,
    }

    for k, v in summary.items():
        print(f'  {k}: {v}')

    return summary

In [55]:
results = []
results.append(evaluate_ultralytics_model('YOLOv8n-seg',  YOLOV8_WEIGHTS, split='test', conf=0.05))
results.append(evaluate_ultralytics_model('YOLO11n-seg',  YOLO11_WEIGHTS, split='test', conf=0.01))

summary_df = pd.DataFrame(results)
out_csv = METRICS_OUT / 'ultralytics_eval_summary.csv'
summary_df.to_csv(out_csv, index=False)
print('\nSaved to:', out_csv)
display(summary_df)


YOLOv8n-seg: evaluating 17 images from test at conf=0.05
  Model: YOLOv8n-seg
  Split: test
  Conf threshold: 0.05
  Num Images: 17
  Num Detected: 12
  Mean Mask IoU: 0.5535
  Mask Precision: 0.5583
  Mask Recall: 0.6407
  Centroid Error (px): 65.13
  Latency ms/img: 231.78
  FPS: 4.3
  Mask mAP50 (training csv): 0.99346
  Mask mAP50-95 (training csv): 0.81832
  Box mAP50 (training csv): 0.99346
  Box mAP50-95 (training csv): 0.85402

YOLO11n-seg: evaluating 17 images from test at conf=0.01
  Model: YOLO11n-seg
  Split: test
  Conf threshold: 0.01
  Num Images: 17
  Num Detected: 17
  Mean Mask IoU: 0.7536
  Mask Precision: 0.8284
  Mask Recall: 0.802
  Centroid Error (px): 79.39
  Latency ms/img: 264.92
  FPS: 3.8
  Mask mAP50 (training csv): 0.995
  Mask mAP50-95 (training csv): 0.80833
  Box mAP50 (training csv): 0.995
  Box mAP50-95 (training csv): 0.84524

Saved to: /content/drive/MyDrive/E-waste Battery Extraction CV/04_metrics_and_visualisations/ultralytics_eval_summary.csv


,Model,Split,Conf threshold,Num Images,Num Detected,Mean Mask IoU,Mask Precision,Mask Recall,Centroid Error (px),Latency ms/img,FPS,Mask mAP50 (training csv),Mask mAP50-95 (training csv),Box mAP50 (training csv),Box mAP50-95 (training csv)
0,YOLOv8n-seg,test,0.05,17,12,0.5535,0.5583,0.6407,65.13,231.78,4.3,0.99346,0.81832,0.99346,0.85402
1,YOLO11n-seg,test,0.01,17,17,0.7536,0.8284,0.8020,79.39,264.92,3.8,0.99500,0.80833,0.99500,0.84524
